<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/pill_imprint_ocr_colab_easyOCR_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pill Imprint OCR Colab Notebook

Pilliot 15k pill imprint OCR Colab notebook source.

흐름:
1. pilliot_15k_v1_final.zip 압축 해제
2. manifest 로드 및 OCR 후보 라벨 정리
3. Train/Inference skew를 막기 위한 공통 전처리와 학습 전용 증강 분리
4. Synthetic data 생성용 scaffold 제공
5. bbox 기준 알약 crop 및 OCR용 2차 전처리
6. EasyOCR / PaddleOCR baseline 실행
7. OCR + 속성 기반 DB 매핑 후보 점수화 scaffold
8. metrics와 실패 케이스 확인


## 0. Colab setup


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip -q install easyocr opencv-python-headless pandas numpy matplotlib tqdm rapidfuzz pillow-heif
!pip -q install albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 147.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 28.1 MB/s eta 0:00:00


In [4]:
from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import pandas as pd
from rapidfuzz.distance import Levenshtein
from tqdm.auto import tqdm


## 1. Paths


In [12]:
ZIP_PATH = "/content/drive/MyDrive/Pillot/dataset/pilliot_15k_optimized_leakage_free_split.zip"
!unzip -q "{ZIP_PATH}" -d /content

In [13]:
DATA_ROOT = Path("/content/pilliot_15k_optimized_leakage_free_split")

MANIFEST_DIR = DATA_ROOT / "manifests"
IMAGE_ROOT = DATA_ROOT / "images"

WORK_DIR = Path("/content/pillot_ocr_work")
CROP_DIR = WORK_DIR / "crops"
RESULT_DIR = WORK_DIR / "results"

for directory in [WORK_DIR, CROP_DIR, RESULT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## 2. Manifest loading


In [19]:
def load_manifest(data_root: Path, manifest_name: str = "bbox_manifest_all_with_attributes.csv") -> pd.DataFrame:
    """manifest CSV를 읽고 필수 컬럼이 있는지 확인합니다."""
    manifest_path = data_root / "manifests" / manifest_name
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest_path}")

    df = pd.read_csv(manifest_path)
    required = {"dataset_type", "split_type", "image_file"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {manifest_path.name}: {sorted(missing)}")
    return df


# OCR 정답에서 비문자 라벨 제외
IGNORE_IMPRINT_TOKENS = {
    "",
    "NAN",
    "NONE",
    "NULL",
    "마크",
    "분할선",
    "없음",
    "무",
    "-",
}


def normalize_imprint(text: object, keep_separator: bool = False) -> str:
    """각인 라벨/예측값을 비교 가능한 형태로 정규화"""
    if pd.isna(text):
        return ""

    text = str(text).strip()
    if text.upper() in IGNORE_IMPRINT_TOKENS:
        return ""

    text = text.upper()
    text = re.sub(r"\s+", "", text)
    text = text.replace("분할선", "")


    allowed = r"[^0-9A-Z가-힣+\-/|]" if keep_separator else r"[^0-9A-Z가-힣+\-/]"
    text = re.sub(allowed, "", text)

    if text.upper() in IGNORE_IMPRINT_TOKENS:
        return ""
    return text


def build_target_candidates(row: pd.Series) -> list[str]:
    """
    이미지 한 장은 앞면 또는 뒷면 하나만 보임.
    따라서 print_front와 print_back 두 개 열을 후보 리스트로 관리
    """
    front = normalize_imprint(row.get("print_front", ""))
    back = normalize_imprint(row.get("print_back", ""))

    candidates = []
    for text in [front, back]:
        if text and text not in candidates:
            candidates.append(text)
    return candidates


def build_target_text(row: pd.Series) -> str:
    """화면 확인용 후보 문자열. 평가는 후보 중 하나와 맞는지로 처리"""
    return "/".join(build_target_candidates(row))


def normalize_prediction(text: object) -> str:
    return normalize_imprint(text, keep_separator=True)


def filter_ocr_candidates(df: pd.DataFrame, split: Optional[str] = None, limit: Optional[int] = None, seed: int = 42) -> pd.DataFrame:
    """OCR 평가 후보만 골라 target_candidates / target_text 컬럼을 추가"""
    out = df.copy()

    if "for_ocr_eval_candidate" in out.columns:
        out = out[out["for_ocr_eval_candidate"].astype(str).str.lower().eq("true")]
    if "has_print" in out.columns:
        out = out[out["has_print"].astype(str).str.lower().eq("true")]
    if split:
        out = out[out["split_type"].eq(split)]

    out["target_text_front"] = out["print_front"].map(normalize_imprint) if "print_front" in out.columns else ""
    out["target_text_back"] = out["print_back"].map(normalize_imprint) if "print_back" in out.columns else ""
    out["target_candidates"] = out.apply(build_target_candidates, axis=1)
    out["target_text"] = out["target_candidates"].map(lambda xs: "/".join(xs))
    out = out[out["target_text"].ne("")]

    if limit:
        import random
        random.seed(seed)
        groups = out.groupby("item_seq")
        all_kinds = list(groups.groups.keys())
        random.shuffle(all_kinds)

        selected, total = [], 0
        for kind in all_kinds:
            count = len(groups.get_group(kind))
            if total + count > limit:
                continue
            selected.append(kind)
            total += count
            if total >= limit * 0.9:
                break
        out = out[out["item_seq"].isin(selected)]

    return out.reset_index(drop=True)


df_all = load_manifest(DATA_ROOT)
df_eval = filter_ocr_candidates(df_all, split=None, limit=500)
print(df_all.shape, df_eval.shape)
display(df_eval.head(20))


/tmp/ipykernel_754/2131662105.py:7: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(manifest_path)


(23517, 40) (471, 44)


,sample_pack,sample_part,detection_source,dataset_type,split_type,label_zip_name,image_zip_name,image_gcs_zip,json_file,image_file,...,relative_image_path,image_exists,component_id,split,pack_relative_image_path,pack_name,target_text_front,target_text_back,target_candidates,target_text
0,zip12_max50_print_required,single_12k,single_12k,single,train,TL_48_단일.zip,TS_48_단일.zip,gs://pilliot-raw-data-2026/00_original_zip/aih...,K-019235_json/K-019235_0_0_0_0_75_000_200.json,K-019235_0_0_0_0_75_000_200.png,...,images/single/TS_48_단일/K-019235_0_0_0_0_75_000...,True,comp_00281,train,images/train/single/TS_48_단일/K-019235_0_0_0_0_...,pilliot_15k_optimized_leakage_free_split,HMHM,,[HMHM],HMHM
1,zip12_max50_print_required,single_12k,single_12k,single,train,TL_48_단일.zip,TS_48_단일.zip,gs://pilliot-raw-data-2026/00_original_zip/aih...,K-019235_json/K-019235_0_0_0_0_75_060_200.json,K-019235_0_0_0_0_75_060_200.png,...,images/single/TS_48_단일/K-019235_0_0_0_0_75_060...,True,comp_00281,train,images/train/single/TS_48_단일/K-019235_0_0_0_0_...,pilliot_15k_optimized_leakage_free_split,HMHM,,[HMHM],HMHM
2,zip12_max50_print_required,single_12k,single_12k,single,train,TL_48_단일.zip,TS_48_단일.zip,gs://pilliot-raw-data-2026/00_original_zip/aih...,K-019235_json/K-019235_0_0_0_0_75_120_200.json,K-019235_0_0_0_0_75_120_200.png,...,images/single/TS_48_단일/K-019235_0_0_0_0_75_120...,True,comp_00281,train,images/train/single/TS_48_단일/K-019235_0_0_0_0_...,pilliot_15k_optimized_leakage_free_split,HMHM,,[HMHM],HMHM
3,zip12_max50_print_required,single_12k,single_12k,single,train,TL_48_단일.zip,TS_48_단일.zip,gs://pilliot-raw-data-2026/00_original_zip/aih...,K-019235_json/K-019235_0_0_0_0_75_280_200.json,K-019235_0_0_0_0_75_280_200.png,...,images/single/TS_48_단일/K-019235_0_0_0_0_75_280...,True,comp_00281,train,images/train/single/TS_48_단일/K-019235_0_0_0_0_...,pilliot_15k_optimized_leakage_free_split,HMHM,,[HMHM],HMHM
4,zip12_max50_print_required,single_12k,single_12k,single,train,TL_48_단일.zip,TS_48_단일.zip,gs://pilliot-raw-data-2026/00_original_zip/aih...,K-019235_json/K-019235_0_0_0_0_75_300_200.json,K-019235_0_0_0_0_75_300_200.png,...,images/single/TS_48_단일/K-019235_0_0_0_0_75_300...,True,comp_00281,train,images/train/single/TS_48_단일/K-019235_0_0_0_0_...,pilliot_15k_optimized_leakage_free_split,HMHM,,[HMHM],HMHM
5,zip12_max50_print_required,single_12k,single_12k,single,train,TL_48_단일.zip,TS_48_단일.zip,gs://pilliot-raw-data-2026/00_original_zip/aih...,K-019235_json/K-019235_0_0_0_0_90_000_200.json,K-019235_0_0_0_0_90_000_200.png,...,images/single/TS_48_단일/K-019235_0_0_0_0_90_000...,True,comp_00281,train,images/train/single/TS_48_단일/K-019235_0_0_0_0_...,pilliot_15k_optimized_leakage_free_split,HMHM,,[HMHM],HMHM
6,zip12_max50_print_required,single_12k,single_12k,single,train,TL_48_단일.zip,TS_48_단일.zip,gs://pilliot-raw-data-2026/00_original_zip/aih...,K-019235_json/K-019235_0_0_0_0_90_140_200.json,K-019235_0_0_0_0_90_140_200.png,...,images/single/TS_48_단일/K-019235_0_0_0_0_90_140...,True,comp_00281,train,images/train/single/TS_48_단일/K-019235_0_0_0_0_...,pilliot_15k_optimized_leakage_free_split,HMHM,,[HMHM],HMHM
7,zip12_max50_print_required,single_12k,single_12k,single,train,TL_48_단일.zip,TS_48_단일.zip,gs://pilliot-raw-data-2026/00_original_zip/aih...,K-019235_json/K-019235_0_0_0_0_90_160_200.json,K-019235_0_0_0_0_90_160_200.png,...,images/single/TS_48_단일/K-019235_0_0_0_0_90_160...,True,comp_00281,train,images/train/single/TS_48_단일/K-019235_0_0_0_0_...,pilliot_15k_optimized_leakage_free_split,HMHM,,[HMHM],HMHM
8,zip12_max50_print_required,single_12k,single_12k,single,train,TL_48_단일.zip,TS_48_단일.zip,gs://pilliot-raw-data-2026/00_original_zip/aih...,K-019235_json/K-019235_0_0_0_0_90_200_200.json,K-019235_0_0_0_0_90_200_200.png,...,images/single/TS_48_단일/K-019235_0_0_0_0_90_200...,True,comp_00281,train,images/train/single/TS_48_단일/K-019235_0_0_0_0_...,pilliot_15k_optimized_leakage_free_split,HMHM,,[HMHM],HMHM
9,zip12_max50_print_required,single_12k,single_12k,single,train,TL_48_단일.zip,

ex) 이미지에 CI or 20 둘 중 하나 인식하면 정답

## 3. Image path resolving


In [28]:
def _zip_stem(name: str) -> str:
    return Path(str(name)).stem


def resolve_image_path(row: pd.Series, data_root: Path = DATA_ROOT) -> Path:
    """
    images / {split_type} / {dataset_type} / {image_zip_stem} / {image_file}
    예) images/train/single/TS_43_단일/K-....png
    """
    split_type     = str(row["split_type"])              # train / val
    dataset_type   = str(row["dataset_type"])            # single / combination
    image_zip_stem = _zip_stem(row["image_zip_name"])    # TS_43_단일.zip → TS_43_단일
    image_file     = str(row["image_file"])              # K-....png
    return data_root / "images" / split_type / dataset_type / image_zip_stem / image_file

def report_missing_images(df: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    """manifest와 실제 이미지 파일 연결이 되는지 빠르게 확인"""
    paths = df.apply(resolve_image_path, axis=1)
    missing_mask = [not p.exists() for p in paths]
    missing = df.loc[missing_mask, ["dataset_type", "split_type", "image_file"]].copy()
    missing["expected_path"] = [str(p) for p, m in zip(paths, missing_mask) if m]
    print(f"missing images: {len(missing)} / {len(df)}")
    return missing.head(n)


display(report_missing_images(df_eval))


missing images: 184 / 471


,dataset_type,split_type,image_file,expected_path
50,single,train,K-036213_0_2_0_0_90_200_200.png,/content/pilliot_15k_optimized_leakage_free_sp...
51,single,train,K-036213_0_2_0_1_75_120_200.png,/content/pilliot_15k_optimized_leakage_free_sp...
52,single,train,K-036213_0_2_0_2_75_280_200.png,/content/pilliot_15k_optimized_leakage_free_sp...
53,single,train,K-036213_0_2_0_2_90_020_200.png,/content/pilliot_15k_optimized_leakage_free_sp...
54,single,train,K-036213_0_2_1_0_90_000_200.png,/content/pilliot_15k_optimized_leakage_free_sp...
55,single,train,K-036213_0_2_1_0_90_060_200.png,/content/pilliot_15k_optimized_leakage_free_sp...
56,single,train,K-036213_0_2_1_1_75_120_200.png,/content/pilliot_15k_optimized_leakage_free_sp...
57,single,train,K-036213_0_2_1_1_75_200_200.png,/content/pilliot_15k_optimized_leakage_free_sp...
58,single,train,K-036213_0_2_1_1_75_240_200.png,/content/pilliot_15k_optimized_leakage_free_sp...
59,single,train,K-036213_0_2_1_1_75_280_200.png,/content/pilliot_15k_optimized_leakage_free_sp...


## 4. 공통 전처리


In [29]:
# 설계 원칙:
# - 공통 전처리: 학습과 추론 모두 적용. 데이터 분포를 맞춰 Train-Inference Skew를 줄임
# - 학습 전용 증강: 공통 전처리 이후에만 적용.
# - 추론 전용 Reject: 사용자 입력 품질이 너무 낮을 때만 차단. training에서는 제외
# - CLAHE: 전체 이미지가 아니라, detection 후 crop된 알약에 대한 2차 OCR 전처리에서 수행

@dataclass
class CommonPreprocessConfig:
    """Detection 학습/추론에 공통으로 적용할 전처리 설정"""

    apply_awb: bool = True
    denoise_method: str = "bilateral"  # "none", "bilateral", "nlm"
    bilateral_d: int = 5
    bilateral_sigma_color: int = 35
    bilateral_sigma_space: int = 35
    nlm_h: int = 3
    target_size: int = 640
    pad_color: tuple[int, int, int] = (114, 114, 114)
    normalize_pixels: bool = False


@dataclass
class InferenceQualityConfig:
    """사용자 입력 이미지 reject 기준(학습 데이터에는 적용 X)"""

    min_laplacian_var: float = 35.0
    min_brightness: float = 35.0
    max_brightness: float = 230.0


def read_bgr(path: Path) -> np.ndarray:
    """
    이미지 포맷을 BGR 3채널 배열로 통일.
    PNG의 알파 채널은 제거되고, HEIC는 pillow-heif가 설치되어 있으면 읽을 수 있음.
    """
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in {".heic", ".heif"}:
        try:
            import pillow_heif
            from PIL import Image

            pillow_heif.register_heif_opener()
            rgb = np.array(Image.open(path).convert("RGB"))
            return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        except ImportError as exc:
            raise ImportError("HEIC 이미지를 읽으려면 `pip install pillow-heif`가 필요합니다.") from exc

    image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if image is None:
        raise FileNotFoundError(path)

    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    if image.shape[2] == 4:
        return cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)
    return image


def gray_world_awb_bgr(image_bgr: np.ndarray) -> np.ndarray:
    """Gray World 방식의 간단한 오토 화이트 밸런스"""
    image = image_bgr.astype(np.float32)
    channel_means = image.reshape(-1, 3).mean(axis=0)
    gray_mean = channel_means.mean()
    scale = gray_mean / np.maximum(channel_means, 1e-6)
    balanced = image * scale
    return np.clip(balanced, 0, 255).astype(np.uint8)


def denoise_bgr(image_bgr: np.ndarray, cfg: CommonPreprocessConfig) -> np.ndarray:
    """
    OCR 각인을 지우지 않도록 약하게 노이즈 제거.
    강한 Gaussian Blur는 각인과 외곽선을 흐리게 만들 수 있어 사용하지 않음.
    """
    if cfg.denoise_method == "none":
        return image_bgr
    if cfg.denoise_method == "bilateral":
        return cv2.bilateralFilter(
            image_bgr,
            d=cfg.bilateral_d,
            sigmaColor=cfg.bilateral_sigma_color,
            sigmaSpace=cfg.bilateral_sigma_space,
        )
    if cfg.denoise_method == "nlm":
        return cv2.fastNlMeansDenoisingColored(image_bgr, None, cfg.nlm_h, cfg.nlm_h, 7, 21)
    raise ValueError(f"Unknown denoise_method: {cfg.denoise_method}")


def letterbox_bgr(
    image_bgr: np.ndarray,
    target_size: int = 640,
    pad_color: tuple[int, int, int] = (114, 114, 114),
) -> tuple[np.ndarray, float, tuple[int, int]]:
    """
    비율을 유지한 채 target_size 정사각형으로 맞춤.
    return: letterbox 이미지, resize 비율, 좌상단 padding(dx, dy)
    """
    h, w = image_bgr.shape[:2]
    scale = min(target_size / h, target_size / w)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))

    resized = cv2.resize(image_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)
    canvas = np.full((target_size, target_size, 3), pad_color, dtype=np.uint8)

    dx = (target_size - new_w) // 2
    dy = (target_size - new_h) // 2
    canvas[dy : dy + new_h, dx : dx + new_w] = resized
    return canvas, scale, (dx, dy)


def common_preprocess_bgr(
    image_bgr: np.ndarray,
    cfg: CommonPreprocessConfig = CommonPreprocessConfig(),
    do_letterbox: bool = False,
) -> np.ndarray | tuple[np.ndarray, float, tuple[int, int]]:
    """
    학습/추론 공통 전처리
    기본 반환은 BGR uint8이며, 모델 입력 직전에만 RGB/float 변환을 권장.
    """
    out = image_bgr
    if cfg.apply_awb:
        out = gray_world_awb_bgr(out)
    out = denoise_bgr(out, cfg)

    if do_letterbox:
        out, scale, pad = letterbox_bgr(out, cfg.target_size, cfg.pad_color)
        if cfg.normalize_pixels:
            out = out.astype(np.float32) / 255.0
        return out, scale, pad

    if cfg.normalize_pixels:
        out = out.astype(np.float32) / 255.0
    return out


def to_detection_input_rgb_float(image_bgr: np.ndarray) -> np.ndarray:
    """Detection 모델 입력 직전에만 RGB + 0~1 float로 변환"""
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return rgb.astype(np.float32) / 255.0


def check_inference_quality(image_bgr: np.ndarray, cfg: InferenceQualityConfig = InferenceQualityConfig()) -> dict:
    """
    사용자 입력 이미지 품질 검증.
    학습에는 적용하지 않고, 실제 서비스 추론 전에만 reject 용도로 사용.
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    brightness = float(hsv[:, :, 2].mean())

    reasons = []
    if blur_score < cfg.min_laplacian_var:
        reasons.append("blur")
    if brightness < cfg.min_brightness:
        reasons.append("underexposed")
    if brightness > cfg.max_brightness:
        reasons.append("overexposed")

    return {
        "accept": len(reasons) == 0,
        "reasons": reasons,
        "blur_score": blur_score,
        "brightness": brightness,
    }


def build_training_augmentation():
    """
    학습 전용 증강(추론에는 절대 적용X)
    OCR 보존을 위해 Flip/Mirror 계열은 넣지 않는 것을 증강 원칙으로 함
    """
    import albumentations as A

    return A.Compose(
        [
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
            A.MotionBlur(blur_limit=3, p=0.15),
            A.GaussNoise(var_limit=(5.0, 25.0), p=0.25),
            A.CoarseDropout(
                max_holes=2,
                max_height=48,
                max_width=48,
                min_holes=1,
                fill_value=114,
                p=0.15,
            ),
            A.Rotate(limit=12, border_mode=cv2.BORDER_CONSTANT, value=(114, 114, 114), p=0.35),
        ],
        bbox_params=A.BboxParams(format="pascal_voc", label_fields=["class_labels"], min_visibility=0.2),
    )


def apply_training_augmentation(image_bgr: np.ndarray, bbox_xyxy: list[float]) -> tuple[np.ndarray, list[float]]:
    """
    학습 전용 증강 예시.
    입력 bbox는 [x1, y1, x2, y2] 형식이며, 추론 파이프라인에서는 호출하지 않음.
    """
    aug = build_training_augmentation()
    transformed = aug(
        image=image_bgr,
        bboxes=[bbox_xyxy],
        class_labels=["pill"],
    )
    new_bbox = list(transformed["bboxes"][0]) if transformed["bboxes"] else bbox_xyxy
    return transformed["image"], new_bbox


def preprocess_for_detection_train(
    image_bgr: np.ndarray,
    bbox_xyxy: Optional[list[float]] = None,
    cfg: CommonPreprocessConfig = CommonPreprocessConfig(),
    use_augmentation: bool = True,
) -> tuple[np.ndarray, Optional[list[float]]]:
    """
    Detection 학습 파이프라인
    공통 전처리 후 학습 전용 증강을 적용하고, 마지막에 모델 입력 크기로 letterbox 처리
    """
    image_bgr = common_preprocess_bgr(image_bgr, cfg, do_letterbox=False)

    if use_augmentation and bbox_xyxy is not None:
        image_bgr, bbox_xyxy = apply_training_augmentation(image_bgr, bbox_xyxy)

    image_bgr, scale, (dx, dy) = letterbox_bgr(image_bgr, cfg.target_size, cfg.pad_color)
    if bbox_xyxy is not None:
        x1, y1, x2, y2 = bbox_xyxy
        bbox_xyxy = [x1 * scale + dx, y1 * scale + dy, x2 * scale + dx, y2 * scale + dy]

    return image_bgr, bbox_xyxy


def preprocess_for_detection_inference(
    image_bgr: np.ndarray,
    cfg: CommonPreprocessConfig = CommonPreprocessConfig(),
    quality_cfg: InferenceQualityConfig = InferenceQualityConfig(),
) -> tuple[Optional[np.ndarray], dict]:
    """
    Detection 추론 파이프라인
    먼저 품질을 검사하고, 통과한 이미지만 공통 전처리 + letterbox를 적용
    """
    quality = check_inference_quality(image_bgr, quality_cfg)
    if not quality["accept"]:
        return None, quality

    image_bgr, scale, pad = common_preprocess_bgr(image_bgr, cfg, do_letterbox=True)
    quality["scale"] = scale
    quality["pad"] = pad
    return image_bgr, quality


## 5. bbox 기반 알약 crop 생성 및 2차 OCR 전처리


In [34]:
@dataclass
class CropConfig:
    """OCR 입력 crop 생성 설정입니다."""

    margin_ratio: float = 0.16
    min_size: int = 96
    target_size: int = 384
    use_bbox: bool = True
    apply_common_preprocess: bool = True


def crop_with_bbox(image_bgr: np.ndarray, row: pd.Series, cfg: CropConfig) -> np.ndarray:
    """manifest bbox에 margin을 더해 알약 영역을 잘라냅니다."""
    h, w = image_bgr.shape[:2]
    if not cfg.use_bbox or any(pd.isna(row.get(k)) for k in ["bbox_x", "bbox_y", "bbox_w", "bbox_h"]):
        return image_bgr

    x, y, bw, bh = [float(row[k]) for k in ["bbox_x", "bbox_y", "bbox_w", "bbox_h"]]
    margin = cfg.margin_ratio * max(bw, bh)
    x1 = max(0, int(round(x - margin)))
    y1 = max(0, int(round(y - margin)))
    x2 = min(w, int(round(x + bw + margin)))
    y2 = min(h, int(round(y + bh + margin)))

    crop = image_bgr[y1:y2, x1:x2]
    if min(crop.shape[:2]) < cfg.min_size:
        return image_bgr
    return crop


def enhance_crop_for_ocr_bgr(image_bgr: np.ndarray, target_size: int = 384) -> np.ndarray:
    """
    OCR 전용 2차 전처리입니다.
    BGR crop을 grayscale 1채널로 바꾸고, CLAHE로 음각/양각 대비를 올립니다.
    EasyOCR/PaddleOCR 입력 호환을 위해 마지막에는 grayscale을 BGR 3채널로 복제합니다.
    """
    h, w = image_bgr.shape[:2]
    scale = target_size / max(h, w)
    if scale != 1:
        image_bgr = cv2.resize(image_bgr, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_CUBIC)

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)


def make_crop(
    row: pd.Series,
    crop_cfg: CropConfig = CropConfig(),
    preprocess_cfg: CommonPreprocessConfig = CommonPreprocessConfig(),
) -> np.ndarray:
    """이미지 로드 → 공통 전처리 → bbox crop → OCR용 대비 보정 순서로 crop을 만듭니다."""
    path = resolve_image_path(row)
    image_bgr = read_bgr(path)

    if crop_cfg.apply_common_preprocess:
        image_bgr = common_preprocess_bgr(image_bgr, preprocess_cfg, do_letterbox=False)

    crop_bgr = crop_with_bbox(image_bgr, row, crop_cfg)
    return enhance_crop_for_ocr_bgr(crop_bgr, crop_cfg.target_size)


def save_eval_crops(df: pd.DataFrame, out_dir: Path = CROP_DIR, limit: Optional[int] = None) -> pd.DataFrame:
    """OCR 평가용 crop 이미지를 저장하고 crop_path를 manifest row에 붙입니다."""
    rows = df.head(limit).copy() if limit else df.copy()
    records = []

    for idx, row in tqdm(rows.iterrows(), total=len(rows), desc="cropping"):
        try:
            crop_bgr = make_crop(row)

            # image_file이 "TS_43_단일/K-....png" 형태라 '/'를 '_'로 치환
            safe_name = str(row["image_file"]).replace("/", "_").replace("\\", "_")
            out_path = out_dir / f"{idx:06d}_{safe_name}"
            cv2.imwrite(str(out_path), crop_bgr)

            record = row.to_dict()
            record["crop_path"] = str(out_path)
            records.append(record)
        except Exception as exc:
            print(f"[skip] {row.get('image_file')}: {exc}")

    return pd.DataFrame(records)


df_crops = save_eval_crops(df_eval)
display(df_crops[["image_file", "target_text_front", "target_text_back", "target_text", "crop_path"]].head())


cropping:   0%|          | 0/471 [00:00<?, ?it/s]

[skip] K-036213_0_2_0_0_90_200_200.png: /content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_0_0_90_200_200.png
[skip] K-036213_0_2_0_1_75_120_200.png: /content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_0_1_75_120_200.png
[skip] K-036213_0_2_0_2_75_280_200.png: /content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_0_2_75_280_200.png
[skip] K-036213_0_2_0_2_90_020_200.png: /content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_0_2_90_020_200.png
[skip] K-036213_0_2_1_0_90_000_200.png: /content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_1_0_90_000_200.png
[skip] K-036213_0_2_1_0_90_060_200.png: /content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_1_0_90_060_200.png
[skip] K-036213_0_2_1_1_75_120_200.png: /content/pilliot_15k_optimized_leakage_free_split/imag

,image_file,target_text_front,target_text_back,target_text,crop_path
0,K-019235_0_0_0_0_75_000_200.png,HMHM,,HMHM,/content/pillot_ocr_work/crops/000000_K-019235...
1,K-019235_0_0_0_0_75_060_200.png,HMHM,,HMHM,/content/pillot_ocr_work/crops/000001_K-019235...
2,K-019235_0_0_0_0_75_120_200.png,HMHM,,HMHM,/content/pillot_ocr_work/crops/000002_K-019235...
3,K-019235_0_0_0_0_75_280_200.png,HMHM,,HMHM,/content/pillot_ocr_work/crops/000003_K-019235...
4,K-019235_0_0_0_0_75_300_200.png,HMHM,,HMHM,/content/pillot_ocr_work/crops/000004_K-019235...


In [35]:
# 1. df_crops 상태 확인
print("shape:", df_crops.shape)
print("columns:", df_crops.columns.tolist())

# 2. 비어있으면 df_eval 상태도 확인
print("\ndf_eval shape:", df_eval.shape)
print("df_eval columns:", df_eval.columns.tolist())

shape: (287, 45)
columns: ['sample_pack', 'sample_part', 'detection_source', 'dataset_type', 'split_type', 'label_zip_name', 'image_zip_name', 'image_gcs_zip', 'json_file', 'image_file', 'dl_mapping_code', 'item_seq', 'dl_name', 'drug_shape', 'color_class1', 'color_class2', 'form_code_name', 'print_front', 'print_back', 'line_front', 'line_back', 'shape_group', 'color_group', 'line_binary', 'has_print', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'area', 'yolo_class_id', 'yolo_class_name', 'dl_company', 'local_image_path', 'relative_image_path', 'image_exists', 'component_id', 'split', 'pack_relative_image_path', 'pack_name', 'target_text_front', 'target_text_back', 'target_candidates', 'target_text', 'crop_path']

df_eval shape: (471, 44)
df_eval columns: ['sample_pack', 'sample_part', 'detection_source', 'dataset_type', 'split_type', 'label_zip_name', 'image_zip_name', 'image_gcs_zip', 'json_file', 'image_file', 'dl_mapping_code', 'item_seq', 'dl_name', 'drug_shape', 'color_class1', 'col

In [37]:
# except에 오류 내용 출력 추가해서 다시 실행
def save_eval_crops(df: pd.DataFrame, out_dir: Path = CROP_DIR, limit: Optional[int] = None) -> pd.DataFrame:
    rows = df.head(limit).copy() if limit else df.copy()
    records = []
    skip_reasons = []  # 추가

    for idx, row in tqdm(rows.iterrows(), total=len(rows), desc="cropping"):
        try:
            crop_bgr = make_crop(row)
            safe_name = str(row["image_file"]).replace("/", "_").replace("\\", "_")
            out_path = out_dir / f"{idx:06d}_{safe_name}"
            cv2.imwrite(str(out_path), crop_bgr)
            record = row.to_dict()
            record["crop_path"] = str(out_path)
            records.append(record)
        except Exception as exc:
            skip_reasons.append({"image_file": row.get("image_file"), "error": str(exc)})  # 추가

    # skip된 오류 요약 출력
    if skip_reasons:
        skip_df = pd.DataFrame(skip_reasons)
        print(f"\nskip {len(skip_df)}건 — 오류 유형별 요약:")
        print(skip_df["error"].value_counts().head(10))
        print("\n첫 5건 샘플:")
        print(skip_df.head())

    return pd.DataFrame(records)

df_crops = save_eval_crops(df_eval)

cropping:   0%|          | 0/471 [00:00<?, ?it/s]


skip 184건 — 오류 유형별 요약:
error
/content/pilliot_15k_optimized_leakage_free_split/images/train/combination/TS_1_조합/K-001900-003544-016551-029451_0_2_0_2_90_000_200.png    2
/content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_0_1_75_120_200.png                             1
/content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_0_0_90_200_200.png                             1
/content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_0_2_90_020_200.png                             1
/content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_1_0_90_000_200.png                             1
/content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_1_0_90_060_200.png                             1
/content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_0_2_75_280_200.png              

In [38]:
# 실제로 파일이 없는 건지 확인
from pathlib import Path

sample_path = Path("/content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일/K-036213_0_2_0_0_90_200_200.png")
print("파일 존재:", sample_path.exists())
print("폴더 존재:", sample_path.parent.exists())

# 해당 폴더에 실제로 있는 파일들
print("\nTS_76_단일 폴더 내 K-036213 파일 목록:")
folder = Path("/content/pilliot_15k_optimized_leakage_free_split/images/train/single/TS_76_단일")
matches = list(folder.glob("K-036213*"))
print(f"총 {len(matches)}개")
for f in matches[:5]:
    print(f.name)

파일 존재: False
폴더 존재: True

TS_76_단일 폴더 내 K-036213 파일 목록:
총 0개


## 6. EasyOCR prototype


In [40]:
def run_easyocr(df_crops: pd.DataFrame, gpu: bool = True) -> pd.DataFrame:
    """EasyOCR baseline입니다. crop_path 이미지를 읽어 OCR 결과를 저장"""
    import easyocr

    reader = easyocr.Reader(["en", "ko"], gpu=gpu)
    results = []

    for _, row in tqdm(df_crops.iterrows(), total=len(df_crops), desc="easyocr"):
        ocr = reader.readtext(
            row["crop_path"],
            detail=1,
            paragraph=False,
            batch_size=1,
            text_threshold=0.35,
            low_text=0.20,
            link_threshold=0.20,
            decoder="beamsearch",
        )

        pieces = []
        confs = []
        for _, text, conf in ocr:
            text = normalize_prediction(text)
            if text:
                pieces.append(text)
                confs.append(float(conf))

        record = row.to_dict()
        record["pred_text"] = "".join(pieces)
        record["ocr_conf"] = float(np.mean(confs)) if confs else 0.0
        record["raw_ocr"] = repr(ocr)
        results.append(record)

    out = pd.DataFrame(results)
    out.to_csv(RESULT_DIR / "easyocr_results.csv", index=False, encoding="utf-8-sig")
    return out


easyocr_results = run_easyocr(df_crops, gpu=True)
display(easyocr_results[["image_file", "target_text", "pred_text", "ocr_conf"]])


easyocr:   0%|          | 0/287 [00:00<?, ?it/s]

,image_file,target_text,pred_text,ocr_conf
0,K-019235_0_0_0_0_75_000_200.png,HMHM,B바시,0.484442
1,K-019235_0_0_0_0_75_060_200.png,HMHM,HC,0.092694
2,K-019235_0_0_0_0_75_120_200.png,HMHM,서도,0.057006
3,K-019235_0_0_0_0_75_280_200.png,HMHM,,0.000000
4,K-019235_0_0_0_0_75_300_200.png,HMHM,,0.000000
...,...,...,...,...
282,K-013004-038972-041149-053384_0_2_0_2_75_000_2...,이가탄F-------마크/IGATANF--------,브I,0.054262
283,K-013004-019881-041149-053384_0_2_0_2_70_000_2...,이가탄F-------마크/IGATANF--------,브A,0.010302
284,K-013004-019881-023319-030850_0_2_0_2_70_000_2...,이가탄F-------마크/IGATANF--------,I,0.088768
285,K-013004-019881-023319-030850_0_2_0_2_90_000_2...,이가탄F-------마크/IGATANF--------,JUI,0.069357
